# Mosaic AI Agent Framework: Author and deploy a multi-agent system with Genie and DSPy
This notebook demonstrates how to build a multi-agent system using Mosaic AI Agent Framework and [DSPy](https://dspy.ai/), where [Genie](https://www.databricks.com/product/business-intelligence/ai-bi-genie) is one of the agents. In this notebook, you will:

1. Author a multi-agent system using DSPy.
1. Wrap the DSPy agent with MLflow ChatAgent to ensure compatibility with Databricks features.
1. Manually test the multi-agent system's output.
1. Log and deploy the multi-agent system.

## Why use a Genie agent?
Multi-agent systems consist of multiple AI agents working together, each with specialized capabilities. 

Unlike SQL functions which can only run pre-defined queries, Genie has the flexibility to create novel queries to answer user questions.

Due to this, Genie becomes a core component of multi-agent solutions which require answering questions over structured data.

## Prerequisites 

1. Address all TODOs in this notebook.

## Notebook contents

1. **Create Genie Spaces** - Create two Genie Spaces for use with our agent (See Databricks documentation for more details ([AWS](https://docs.databricks.com/aws/en/genie/set-up) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/genie/set-up))). This notebook includes steps to obtain data to use with the Genie data rooms, followed by steps for creating the Genie rooms themselves. You can choose to use your own data if you wish.
1. **Write your agent using DSPy**
1. **Test the agent**
1. **Log the agent as an MLflow model**
1. **Register the model in Unity Catalog**
1. **Deploy the agent**

### Install dependencies

In [0]:
%pip install -qqqq --upgrade dspy mlflow databricks-sdk databricks-agents pydantic uv
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 1. Create Genie Spaces

### Load Marketplace data for Genie Rooms
Skip this section if you have existing data you want to use

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

In [0]:
from databricks.sdk.service.marketplace import ConsumerTerms

def get_current_user()-> str:
  return (dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .userName()
    .get()
    .split("@")[0]
    .replace(".", "_"))

def get_terms_of_service_version(terms_of_service: str)->str:
  return terms_of_service.split("/files/")[1].split("/")[0]

def install_marketplace_tables(listing, catalog_suffix:str) -> None:
  consumer_terms_version = get_terms_of_service_version(listing.detail.terms_of_service)
  w.consumer_installations.create(
    listing_id=listing.id, 
    share_name=listing.summary.share.name, 
    catalog_name=f"{get_current_user()}_{catalog_suffix}",
    accepted_consumer_terms=ConsumerTerms(version=consumer_terms_version)
  )

def delete_marketplace_tables(listing_id:str, catalog_suffix:str) -> None:
  installation = [
      installation
      for installation in w.consumer_installations.list()
      if installation.catalog_name == f"{get_current_user()}_{catalog_suffix}"
  ][0]

  w.consumer_installations.delete(
      installation_id=installation.id,
      listing_id=listing_id,
  )

### Load Retail Catalog from Marketplace

In [0]:
simulated_retail_listing = [marketplace_list for marketplace_list in w.consumer_listings.list() if marketplace_list.summary.name=='Simulated Retail Customer Data'][0]

install_marketplace_tables(listing=simulated_retail_listing, catalog_suffix="us_retail")

# Uncomment to delete the retail tables
# delete_marketplace_tables(listing_id=simulated_retail_listing.id, catalog_suffix="us_retail")

### Load Australia Sales Catalog from Marketplace

In [0]:
australia_sales_listing = [marketplace_list for marketplace_list in w.consumer_listings.list() if marketplace_list.summary.name=='Simulated Australia Sales and Opportunities Data'][0]

install_marketplace_tables(listing=australia_sales_listing, catalog_suffix="australia_sales")

# Uncomment to delete the retail tables
# delete_marketplace_tables(listing_id=australia_sales_listing.id, catalog_suffix="australia_sales")

### Create 2 Genie Data Rooms: 

After running the cells above, you will have the following delta shared catalogs available in your workspace: 
1. `<your_username>_us_retail` - contains US retail data
1. `<your_username>_australia_sales` - contains Australian sales data

**TODO:** Follow the steps given in the official docs ([AWS](https://docs.databricks.com/aws/en/genie/set-up) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/genie/set-up)) to create 2 Genie data rooms:
1. Which contains all the tables in the `{your_username}_us_retail.v01` schema
1. Which contains all the tables in the `{your_username}_australia_sales.v01` schema  

## 2. Write your agent using DSPy

The key parts of our DSPy agent will be:

1. An LLM that we can configure via dspy.configure
2. A dspy.Signature, that defines how the LLM should accomplish the task. A signature structures your inputs and outputs, enforces typing and provides additional instructions. 
3. A dspy.Module to convert the signature into a prompt. Depending on the module, you can enable agents (dspy.ReAct), chain of thought or reasoning (dspy.CoT) or just a simple predict (dspy.Predict). The module's output will be a prediction where we can programatically access the outputs.

To learn more about DSPy, check out the [DSPy docs](https://dspy.ai/).

**TODO**: Fill in the values below and run the cell to generate a config.yaml file. Your agent will access the values from this file. 

You can find your Genie space ID in the URL of each respective Genie space. URLs are of the format: `https://<databricks-instance>/genie/rooms/<space-id>`

In [0]:
%%writefile config.yaml

# You can specify a different llm endpoint if you wish to
llm_endpoint_name: "databricks-meta-llama-3-3-70b-instruct"

australian_sales_genie_space_id: "<your_genie_space_id>"
us_retail_genie_space_id: "<your_genie_space_id>"

sql_warehouse_id: "<your_sql_warehouse_id>"

# For registering your mlflow model
catalog: "<your_catalog>"
schema: "your_schema"
model_name: "<your_model_name>"

Overwriting config.yaml


**TODO:** Assign a PAT Token to DATABRICKS_TOKEN. 

Follow these [instructions](https://docs.databricks.com/aws/en/dev-tools/auth/pat#databricks-personal-access-tokens-for-workspace-users) to generate a PAT Token.

In [0]:
import os

os.environ["DATABRICKS_HOST"] = spark.conf.get("spark.databricks.workspaceUrl")
os.environ["DATABRICKS_TOKEN"] = ''

We start by setting up mlflow experiment path

In [0]:
import mlflow

current_user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()

mlflow_experiment_path = f"/Users/{current_user}/dspy_multi_genie"
mlflow.set_experiment(experiment_name=mlflow_experiment_path)

# Get the experiment ID to use in the next step
experiment_id = mlflow.tracking.fluent._get_experiment_id()
print(experiment_id)

2025/08/06 22:51:53 INFO mlflow.tracking.fluent: Experiment with name '/Users/hanna.moazam@databricks.com/dspy_multi_genie' does not exist. Creating a new experiment.


1305773837050671


We will now use `%%writefile` to create a python file containing our agents code.

In [0]:
%%writefile agent.py

import mlflow
from mlflow.models import ModelConfig
from typing import Any, Generator, Optional
from databricks.sdk.service.dashboards import GenieAPI
import mlflow
from databricks.sdk import WorkspaceClient
from mlflow.entities import SpanType
from mlflow.pyfunc.model import ChatAgent
from mlflow.types.agent import (
    ChatAgentMessage,
    ChatAgentResponse,
    ChatContext,
)
import dspy
import uuid
import datetime
import os

# Autolog DSPy traces to MLflow
mlflow.dspy.autolog()

config_file = "config.yaml"
model_config = mlflow.models.ModelConfig(development_config=config_file)

# Set up DSPy with a Databricks-hosted LLM
LLM_ENDPOINT_NAME = model_config.get("llm_endpoint_name")
lm = dspy.LM(model=f"databricks/{LLM_ENDPOINT_NAME}")
dspy.settings.configure(lm=lm)


######################################
## Create our Genie Selector Signature
######################################
# This signature will be used to determine which Genie Space based on the request and utilize the relevant Genie Space using self-reasoning to select the proper tool.

# Deciding on your module’s inputs, outputs, and their types is the most critical part of creating a signature. This definition forms the contract that other components in your multi-agent pipeline will rely on.

class genie_selector_agent(dspy.Signature):
  """
  Given a question, determine which genie space tool to call, send the exact question to the tool and answer the question given the response from the tool.
  """ 
  # **NOTE** : You can add details to the prompt about each tool to increase the accuracy of the agent. Both the function name and prompt inform the agent what the tool is used for.

  question: str = dspy.InputField()
  response: str = dspy.OutputField() 
  sql_query_output:  list = dspy.OutputField()


#####################################
## Create our DSPY Chat Agent Class
#####################################
class DSPyChatAgent(ChatAgent):     
    def __init__(self):
      self.w = WorkspaceClient(
        host=os.getenv("DATABRICKS_HOST"),
        token=os.getenv("DATABRICKS_TOKEN")
      )

      self.genie_selector_agent = genie_selector_agent

      self.multi_genie_agent = dspy.ReAct(self.genie_selector_agent, 
                                          tools=[self.australian_sales_space, self.us_sales_space], 
                                          max_iters=1)



    #######################################################
    ## Create a Function that will call the 1st Genie Space
    #######################################################

    def australian_sales_space(self, question):
      """This genie space is used to query data about Australia Sales. This genie space takes in a request and returns an answer based on the relevant data."""

      genie_space_id = model_config.get("australian_sales_genie_space_id")

      # Start a conversation
      conversation = self.w.genie.start_conversation_and_wait(
          space_id=genie_space_id,
          content=f"{question} always limit to one result",
          timeout=datetime.timedelta(minutes=20)
      )

      response = self.w.genie.get_message_attachment_query_result(
        space_id=genie_space_id,
        conversation_id=conversation.conversation_id,
        message_id=conversation.message_id,
        attachment_id=conversation.attachments[0].attachment_id
      )

      return response.statement_response.result.data_array
    
    #######################################################
    ## Create a Function that will call the 2nd Genie Space
    #######################################################
    
    def us_sales_space(self, question):
      """This genie space is used to query data about USA Retail Sales. This genie space takes in a request and returns an answer based on the relevant data."""

      genie_space_id = model_config.get("us_retail_genie_space_id")

      # Start a conversation
      conversation = self.w.genie.start_conversation_and_wait(
          space_id=genie_space_id,
          content=f"{question} always limit to one result",
          timeout=datetime.timedelta(minutes=20)
      )

      response = self.w.genie.get_message_attachment_query_result(
        space_id=genie_space_id,
        conversation_id=conversation.conversation_id,
        message_id=conversation.message_id,
        attachment_id=conversation.attachments[0].attachment_id
      )

      return response.statement_response.result.data_array


    def prepare_message_history(self, messages: list[ChatAgentMessage]):
        history_entries = []
        # Assume the last message in the input is the most recent user question.
        for i in range(0, len(messages) - 1, 2):
            history_entries.append({"question": messages[i].content, "answer": messages[i + 1].content})
        return dspy.History(messages=history_entries)

    @mlflow.trace(span_type=SpanType.AGENT)
    def predict(
        self,
        messages: list[ChatAgentMessage],
        context: Optional[ChatContext] = None,
        custom_inputs: Optional[dict[str, Any]] = None,
    ) -> ChatAgentResponse:
        latest_question = messages[-1].content
        response = self.multi_genie_agent(question=latest_question).response
        return ChatAgentResponse(
            messages=[ChatAgentMessage(role="assistant", content=response, id=uuid.uuid4().hex)]
        )

# Set model for logging or interactive testing
from mlflow.models import set_model
AGENT = DSPyChatAgent()
set_model(AGENT)

Overwriting agent.py


## 3. Test the agent
Interact with the agent to test its output. Since this notebook called mlflow.dspy.autolog() you can view the trace for each step the agent takes.

In [0]:
from agent import AGENT

input_example= "Who is the top buyer in Australia?"

response = AGENT.predict({"messages": [{"role": "user", "content": input_example}]})

print(response.messages[0].content)

2025/09/13 17:36:18 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/local_disk0/.ephemeral_nfs/envs/pythonEnv-0d04d3ba-4019-4588-b05c-f1b092c5d60a/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 5: Expected `Message` - serialized value may not be as expected [input_value=Message(content='[[ ## re...er_specific_fields=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [input_value=Choices(finish_reason='st...r_specific_fields=None)), input_type=Choices])"


The top buyer in Australia is Transpacific Systems, with the buyer ID TS4266, and they have spent approximately 5.229032696000002E7.


Trace(trace_id=tr-e976db1acc80dc64eddc08f15d2e2a07)

In [0]:
from agent import AGENT

input_example= "Who is the top buyer in USA?"

response  = AGENT.predict({"messages": [{"role": "user", "content": input_example}]})

print(response.messages[0].content)

The top buyer in the USA is 'ab digital, llc' with a total of '1406196' in sales, and their buyer ID is '31375543'.


Trace(trace_id=tr-a0179d8765d379a1ad5adb2e1e87cf52)

## 4. Log the agent as an MLflow model
Log the agent as code from the `agent.py` file that was written by the previous cell. See [MLflow - Models from Code](https://mlflow.org/docs/latest/ml/model/#models-from-code).

### Enable automatic authentication for Databricks resources
For the most common Databricks resource types, Databricks supports and recommends declaring resource dependencies for the agent upfront during logging. This enables automatic authentication passthrough when you deploy the agent. With automatic authentication passthrough, Databricks automatically provisions, rotates, and manages short-lived credentials to securely access these resource dependencies from within the agent endpoint.

To enable automatic authentication, specify the dependent Databricks resources when calling mlflow.pyfunc.log_model().

In [0]:
import mlflow
from mlflow.models.resources import (
    DatabricksFunction,
    DatabricksGenieSpace,
    DatabricksServingEndpoint,
    DatabricksSQLWarehouse,
)
from pkg_resources import get_distribution

from mlflow.models import ModelConfig
config_file = "config.yaml"
model_config = mlflow.models.ModelConfig(development_config=config_file)

resources = [
    DatabricksServingEndpoint(endpoint_name= model_config.get("llm_endpoint_name")),
    DatabricksGenieSpace(genie_space_id= model_config.get("australian_sales_genie_space_id")),
    DatabricksGenieSpace(genie_space_id= model_config.get("us_retail_genie_space_id")),
    DatabricksSQLWarehouse(warehouse_id= model_config.get("warehouse_id")),
]

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent.py",
        model_config="config.yaml",
        extra_pip_requirements=[
            f"databricks-sdk=={get_distribution('databricks-sdk').version}",
            "dspy",
        ],
        resources=resources,
    )

🔗 View Logged Model at: https://e2-demo-field-eng.cloud.databricks.com/ml/experiments/1305773837049705/models/m-64616d84248144ff9b276c564ddd182e?o=1444828305810485
2025/09/13 17:41:50 INFO mlflow.pyfunc: Predicting on input example to validate output
2025/09/13 17:41:58 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/local_disk0/.ephemeral_nfs/envs/pythonEnv-0d04d3ba-4019-4588-b05c-f1b092c5d60a/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 5: Expected `Message` - serialized value may not be as expected [input_value=Message(content="[[ ## re...er_specific_fields=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [input_value=Choices(finish_reason='st...r_specific_fields=None)), input_type=Choices])"
2025/09/13 17:42:06 WARNING mlflow.utils.environment: Enc

### Pre-deployment agent validation
Before registering and deploying the agent, perform pre-deployment checks using the [mlflow.models.predict()](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.models.html#mlflow.models.predict) API. See Databricks documentation ([AWS](https://docs.databricks.com/aws/en/machine-learning/model-serving/model-serving-debug#validate-inputs) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/machine-learning/model-serving/model-serving-debug#before-model-deployment-validation-checks)).

In [0]:
mlflow.models.predict(
    model_uri=f"runs:/{logged_agent_info.run_id}/agent",
    input_data={"messages": [{"role": "user", "content": input_example}]},
    env_manager="uv",
)

2025/09/13 17:42:22 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


2025/09/13 17:42:24 INFO mlflow.utils.virtualenv: Creating a new environment in /tmp/virtualenv_envs_e2d0b798/mlflow-af61b045d04f3e6950da493ce98f70b91881eb75 with python version 3.12.3 using uv
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: /tmp/virtualenv_envs_e2d0b798/mlflow-af61b045d04f3e6950da493ce98f70b91881eb75
Activate with: source /tmp/virtualenv_envs_e2d0b798/mlflow-af61b045d04f3e6950da493ce98f70b91881eb75/bin/activate
2025/09/13 17:42:25 INFO mlflow.utils.virtualenv: Installing dependencies
Using Python 3.12.3 environment at: /tmp/virtualenv_envs_e2d0b798/mlflow-af61b045d04f3e6950da493ce98f70b91881eb75
Resolved 3 packages in 2ms
Installed 3 packages in 14ms
 + pip==24.0
 + setuptools==74.0.0
 + wheel==0.43.0
Using Python 3.12.3 environment at: /tmp/virtualenv_envs_e2d0b798/mlflow-af61b045d04f3e6950da493ce98f70b91881eb75
Resolved 119 packages in 385ms
Prepared 44 packages in 424ms
Installed 119 packages in 158ms
 + aiohappyeyeballs==2

{"messages": [{"role": "assistant", "content": "The top buyer in the USA is 'statusdigital' from Chicago, IL, with a buyer ID of '21532231' and a total of 876 purchases.", "id": "9c6af605c04747c1a408d63bfc86ca41"}]}

2025/09/13 17:42:47 INFO mlflow.tracing.export.async_export_queue: Flushing the async trace logging queue before program exit. This may take a while...


## 5. Register the Model in Unity Catalog

Update the `catalog`, `schema`, and `model_name` below to register the MLflow model to Unity Catalog.



In [0]:
mlflow.set_registry_uri("databricks-uc")

# TODO: define the catalog, schema, and model name for your UC model.
catalog = model_config.get("catalog")
schema = model_config.get("schema")
model_name = model_config.get("model_name")
UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# register the model to UC
uc_registered_model_info = mlflow.register_model(model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME)

Registered model 'hanna_moazam.dspy.dspy_genie_agent' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/14 [00:00<?, ?it/s]

🔗 Created version '1' of model 'hanna_moazam.dspy.dspy_genie_agent': https://e2-demo-field-eng.cloud.databricks.com/explore/data/models/hanna_moazam/dspy/dspy_genie_agent/version/1?o=1444828305810485


## 6. Deploy the agent

In [0]:
from databricks import agents
import os

agents.deploy(
  UC_MODEL_NAME, 
  uc_registered_model_info.version, 
  tags={"endpointSource": "docs"},
  environment_vars={
    "DATABRICKS_HOST": os.getenv("DATABRICKS_HOST"), 
    "DATABRICKS_TOKEN": os.getenv("DATABRICKS_TOKEN")
  },  
)

## Next steps
After your agent is deployed, you can chat with it in AI playground to perform additional checks, share it with SMEs in your organization for feedback, or embed it in a production application. See Databricks documentation ([AWS](https://docs.databricks.com/en/generative-ai/deploy-agent.html) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/deploy-agent)).